In [ ]:
# 2026-05-15 run against production
from apis_core.apis_metainfo.models import Uri, Collection, CollectionType
from apis_core.apis_entities.models import Place
from apis_core.apis_vocabularies.models import PlaceType, PlacePlaceRelation
from apis_core.apis_relations.models import PlacePlace
from acdh_tei_pyutils.tei import TeiReader
from acdh_tei_pyutils.utils import any_xpath, get_xmlid
from normdata.utils import import_from_normdata
from tqdm import tqdm

In [ ]:
col, _ = Collection.objects.get_or_create(name="Schönberg-Briefe: Universal-Edition und Dreililien")
domain = "schoenberg-ue"
col_type, _ = CollectionType.objects.get_or_create(name="Projekt")
col.description = """
Arnold Schönberg: Briefwechsel mit den Verlagen Universal-Edition und Dreililien. Hrsg. von Katharina Bleier und Therese Muxeneder unter Mitarbeit von Jannik Franz und Philipp Kehrer, Universität für Musik und darstellende Kunst Wien und Arnold Schönberg Center, Wien
<a href="https://www.schoenberg-ue.at">https://www.schoenberg-ue.at</a>
"""
col.collection_type = col_type
col.save()

In [ ]:
source_file_uri = "https://raw.githubusercontent.com/Schoenberg-UE/Schoenberg-Data/refs/heads/main/data/indices/Orte.xml"

In [ ]:
doc = TeiReader(source_file_uri)

In [ ]:
items = doc.any_xpath(".//tei:place[@xml:id]")
url_stub = "https://www.schoenberg-ue.at/ue/places/"
place_type = PlaceType.objects.get(name="Gebäude (K.GBD)")
place_place_type = PlacePlaceRelation.objects.get(name="enthält") 

In [ ]:
with_id = 0
no_id = 0
print(len(items))
for x in tqdm(items, total=len(items)):
    xml_id = get_xmlid(x)
    tmp_uri = f"{url_stub}{xml_id}"
    label = any_xpath(x, "./tei:placeName[@type='reg']")[0].text
    try:
        geonames = any_xpath(x, "./tei:idno[@type='uri']/text()")[0]
        with_id += 1
        entity = import_from_normdata(geonames, "place")
        if entity:
            pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_uri, domain=domain)
            pmb_uri.entity = entity
            pmb_uri.save()
            entity.collection.add(col)
    except IndexError:
        pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_uri, domain=domain)
        entity = pmb_uri.entity
        if entity:
            pass
        else:
            item = {
                "name": label,
                "kind": place_type
            }
            entity = Place.objects.create(**item)
            entity.collection.add(col)
            pmb_uri.entity = entity
            pmb_uri.save()
        parent_place = any_xpath(x, "ancestor::tei:place[1]")[0]
        parent_xml_id = get_xmlid(parent_place)
        parent_url = f"{url_stub}{parent_xml_id}"
        parent_entity = Uri.objects.get(uri=parent_url).entity
        parent_entity = Place.objects.get(id=parent_entity.id)
        pl_entity = Place.objects.get(id=entity.id)
        PlacePlace.objects.get_or_create(related_placea=parent_entity, related_placeb=pl_entity, relation_type=place_place_type)
        geo = any_xpath(x, ".//tei:geo")[0]
        try:
            lat, lng = geo.text.split(", ")
        except AttributeError:
            continue
        pl_entity.lat = float(lat)
        pl_entity.lng = float(lng)
        pl_entity.save()
